<h2><a href="https://leetcode.com/problems/second-highest-salary/">176. Second Highest Salary</a></h2><h3>Medium</h3><hr><div class="sql-schema-wrapper__3VBi"><a class="sql-schema-link__3cEg">SQL Schema<svg viewBox="0 0 24 24" width="1em" height="1em" class="icon__1Md2"><path fill-rule="evenodd" d="M10 6L8.59 7.41 13.17 12l-4.58 4.59L10 18l6-6z"></path></svg></a></div><div><p>Table: <code>Employee</code></p>

<pre>+-------------+------+
| Column Name | Type |
+-------------+------+
| id          | int  |
| salary      | int  |
+-------------+------+
id is the primary key (column with unique values) for this table.
Each row of this table contains information about the salary of an employee.
</pre>

<p>&nbsp;</p>

<p>Write a solution to find&nbsp;the second highest salary from the <code>Employee</code> table. If there is no second highest salary,&nbsp;return&nbsp;<code>null (return&nbsp;None in Pandas)</code>.</p>

<p>The result format is in the following example.</p>

<p>&nbsp;</p>
<p><strong class="example">Example 1:</strong></p>

<pre><strong>Input:</strong> 
Employee table:
+----+--------+
| id | salary |
+----+--------+
| 1  | 100    |
| 2  | 200    |
| 3  | 300    |
+----+--------+
<strong>Output:</strong> 
+---------------------+
| SecondHighestSalary |
+---------------------+
| 200                 |
+---------------------+
</pre>

<p><strong class="example">Example 2:</strong></p>

<pre><strong>Input:</strong> 
Employee table:
+----+--------+
| id | salary |
+----+--------+
| 1  | 100    |
+----+--------+
<strong>Output:</strong> 
+---------------------+
| SecondHighestSalary |
+---------------------+
| null                |
+---------------------+
</pre>
</div>

In [0]:
# Import pandas API for PySpark and types for schema definition
import pyspark.pandas as ps
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Sample employee data as a list of tuples (id, salary)
# employee_data = [(1, 100), (2, 200), (3, 300)]
employee_data = [(1, 100)]

# Define schema for Employee table
employee_schema = StructType([
    StructField('id', IntegerType()),
    StructField('salary', IntegerType())
])

# Create Spark DataFrame from employee data and schema
employee_df = spark.createDataFrame(data=employee_data, schema=employee_schema)

# Convert Spark DataFrame to pandas API DataFrame
employee_df_pandas = employee_df.pandas_api()

# Register DataFrame as a temporary SQL view
employee_df.createOrReplaceTempView("Employee")

# Return pandas API DataFrame
employee_df_pandas

In [0]:
%sql
-- CTE to assign row numbers to distinct salaries in descending order
with cte_employee as (
  select
    salary,
    row_number() over (order by salary desc) as row_num
  from
    Employee
  group by
    salary
)
-- Select the second highest salary if it exists
select
  salary as SecondHighestSalary
from
  cte_employee
where
  row_num = 2
union all
-- Return null if there is no second highest salary
select
  null as SecondHighestSalary
where
  not exists (
    select
      1
    from
      cte_employee
    where
      row_num = 2
  )

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

# Define a window to order salaries in descending order
window = Window.orderBy(F.col("salary").desc())

# Find the second highest salary using dense_rank
second_highest_salary = (
    employee_df.withColumn("rank", F.dense_rank().over(window))
    .filter(F.col("rank") == 2)
    .select(F.col("salary").alias("SecondHighestSalary"))
    .limit(1)  # in case multiple employees share the second highest salary
)

# If no second highest salary exists, return a DataFrame with null
if second_highest_salary.count() == 0:
    schema = StructType([StructField("SecondHighestSalary", IntegerType(), True)])
    second_highest_salary = spark.createDataFrame([(None)], schema=schema)

# Display the result
display(second_highest_salary)